In [1]:
from zigzag import *

In [3]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
# raw = pd.read_csv('./data/ic2509_250906.csv')
raw = pd.read_csv('./data/ic2509_250912.csv')

In [5]:
klines=raw.loc[:,['time','open','high','low','close','volume']]

In [6]:
kline = klines.copy()

In [79]:
kline['zig'] = peak_valley_pivots(kline.close, 0.003, -0.003)

#     获取peak close
peak_temp = (kline.zig == 1) *  kline.close
peak_temp.replace(0,np.nan,inplace=True)
kline['peak'] = peak_temp.ffill()
peak_ref = kline.zig # 初始化一个series 来存放之前的第二个peak
peak_non_zero_mask = kline.zig == 1
peak_non_zero_values = peak_temp[peak_non_zero_mask]
peak_shifted_values = pd.Series([np.nan] + peak_non_zero_values.tolist()[:-1], 
                          index=peak_non_zero_values.index)
peak_ref_2 =peak_ref.mask(peak_ref ==1, peak_shifted_values)

# #     获取valley close
# valley_temp = (kline.zig == -1) *  kline.close
# valley_temp.replace(0,np.nan,inplace=True)
# kline['valley'] = valley_temp.ffill()

# # 将peak valley 整合到一列
# p_v_temp = (kline.zig != 0) * kline.close
# p_v_temp = (kline.zig != 0) * kline.close

In [87]:
peak_temp.notna()

0       False
1        True
2       False
3       False
4       False
        ...  
2095    False
2096    False
2097    False
2098    False
2099     True
Length: 2100, dtype: bool

In [109]:
def get_nth_peak(peak,n):
#     peakc     = peak.copy()
    peak_non_zero_mask = peak_temp.notna()
    peak_non_zero_values = peak[peak_non_zero_mask]
    peak_shifted_values = pd.Series([np.nan]*n + peak_non_zero_values.tolist()[:-n], 
                              index=peak_non_zero_values.index)
    return peak.mask(peak.notna(), peak_shifted_values)
    

In [111]:
get_nth_peak(peak_temp,1).ffill()[100:110]

100    5955.8
101    5955.8
102    5955.8
103    5955.8
104    5955.8
105    5955.8
106    5955.8
107    5955.8
108    5955.8
109    5955.8
dtype: float64

In [74]:
def get_nth_peak3(peak,n):
    peak_non_zero_mask = peak != 0
    peak_non_zero_values = peak[peak_non_zero_mask]
    peak_shifted_values = pd.Series([np.nan]*n + peak_non_zero_values.tolist()[:-n], 
                              index=peak_non_zero_values.index)
    return peak.mask(peak ==1, peak_shifted_values)
    

In [100]:
test = get_nth_peak(peak_temp,1)

In [102]:
test.ffill()

0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
         ...  
2095    7182.4
2096    7182.4
2097    7182.4
2098    7182.4
2099    7190.0
Length: 2100, dtype: float64

In [ ]:
get_nth_peak(peak_temp,1)

In [71]:
def get_nth_peak2(peak,n):
    non_zero_mask = peak!=0
    return peak.where(~non_zero_mask, 
                    pd.Series([np.nan]*n + peak[non_zero_mask].tolist()[:-n], 
                             index=peak[non_zero_mask].index))

In [86]:
get_nth_peak2(peak_temp,1)

0          NaN
1          NaN
2       5910.0
3          NaN
4          NaN
         ...  
2095       NaN
2096       NaN
2097       NaN
2098       NaN
2099       NaN
Length: 2100, dtype: float64

In [54]:
peak_ref_2

0         -1.0
1          NaN
2          0.0
3          0.0
4          0.0
         ...  
2095       0.0
2096       0.0
2097       0.0
2098       0.0
2099    7190.0
Name: zig, Length: 2100, dtype: float64

In [56]:
peak_ref_2

0         -1.0
1          NaN
2          0.0
3          0.0
4          0.0
         ...  
2095       0.0
2096       0.0
2097       0.0
2098       0.0
2099    7190.0
Name: zig, Length: 2100, dtype: float64

In [42]:
peak_temp.iloc[[1,41,75]]

1     5910.0
41    5955.8
75    5911.8
dtype: float64

In [30]:
shifted_values

0          NaN
1          NaN
9       5910.0
41         NaN
69      5955.8
         ...  
2031    7062.0
2064       NaN
2069    7182.4
2085       NaN
2099    7190.0
Length: 193, dtype: float64

In [28]:
# 找出非零值的索引和值
non_zero_mask = s != 0
non_zero_values = s[non_zero_mask]

0          NaN
1       5910.0
2          NaN
3          NaN
4          NaN
         ...  
2095       NaN
2096       NaN
2097       NaN
2098       NaN
2099    7140.0
Length: 2100, dtype: float64

In [15]:
non_zero_mask = p_v_temp!=0

In [17]:
p_v_temp[non_zero_mask]

0       5899.4
1       5910.0
9       5853.0
41      5955.8
69      5889.4
         ...  
2031    7030.8
2064    7182.4
2069    7120.2
2085    7190.0
2099    7140.0
Length: 193, dtype: float64

In [20]:
non_zero_values = p_v_temp[non_zero_mask].values
shifted_values = np.concatenate([[np.nan], non_zero_values[:-1]])

In [21]:
shifted_values

array([   nan, 5899.4, 5910. , 5853. , 5955.8, 5889.4, 5911.8, 5893.2,
       6017.6, 5986. , 6010. , 5970.6, 6072.2, 6040. , 6077. , 6051.2,
       6129. , 6087.8, 6157. , 6123.2, 6177. , 6105. , 6189.8, 6171. ,
       6209.8, 6186.2, 6231.8, 6203.8, 6228.6, 6197.8, 6222.4, 6202.6,
       6222.8, 6179.6, 6250. , 6196. , 6231.4, 6169.8, 6222.2, 6199.6,
       6263. , 6228. , 6272. , 6176.2, 6231.8, 6149.2, 6183. , 6132.6,
       6155.6, 6092.8, 6160.8, 6088.8, 6119.6, 6090.8, 6189.2, 6165. ,
       6274. , 6197.4, 6237.6, 6199.6, 6247.6, 6214.6, 6307.2, 6280. ,
       6337. , 6287.4, 6320.8, 6292.4, 6461. , 6429. , 6457.6, 6361.8,
       6415.2, 6380.2, 6479.2, 6459. , 6535.2, 6513.4, 6574.8, 6538. ,
       6677.8, 6591.2, 6643.2, 6596.6, 6645.8, 6548. , 6596.2, 6570.8,
       6610.2, 6541.4, 6712.6, 6645.4, 6693.6, 6659.8, 6691.4, 6623.6,
       6720. , 6690.6, 6747.6, 6725.2, 6750.8, 6728.4, 6894.2, 6852.4,
       6885.8, 6859.6, 6948.8, 6841.8, 6937.8, 6899.6, 6946. , 6917.2,
      

In [ ]:
p_v_temp=

In [ ]:
result[non_zero_mask] = shifted_values